# 02 – Data Quality & Cleaning

**Input:** data/raw/meals_raw.csv (23,663 meals from fetch_mensa.py)

**Goal:** check data quality, decide which canteens to keep, and clean the data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/meals_raw.csv")
df.shape

(23663, 12)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23663 entries, 0 to 23662
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                23663 non-null  int64  
 1   name              23663 non-null  str    
 2   category          23663 non-null  str    
 3   notes             23663 non-null  str    
 4   canteen_id        23663 non-null  int64  
 5   canteen_name      23663 non-null  str    
 6   city              23663 non-null  str    
 7   date              23663 non-null  str    
 8   prices.students   19768 non-null  float64
 9   prices.employees  19768 non-null  float64
 10  prices.pupils     1331 non-null   float64
 11  prices.others     17996 non-null  float64
dtypes: float64(4), int64(2), str(6)
memory usage: 2.2 MB


In [3]:
df.head()

,id,name,category,notes,canteen_id,canteen_name,city,date,prices.students,prices.employees,prices.pupils,prices.others
0,20807890,Paniertes Seelachsfilet,Fischgericht,"['Dillsoße', 'Fingermöhren', 'Brokkoli', 'Kurk...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,4.10,6.15,NaN,8.15
1,20807891,Frikadelle mit Käse überbacken,Fleischgericht,"['Gebratenes Putensteak', 'Letscho ungarische ...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,3.70,5.95,NaN,7.85
2,20559265,Letscho ungarische Art,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
3,20559266,Fingermöhren,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
4,20559267,Karotten-Kürbis-Gemüse,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75


In [4]:
df.isna().sum()

id                      0
name                    0
category                0
notes                   0
canteen_id              0
canteen_name            0
city                    0
date                    0
prices.students      3895
prices.employees     3895
prices.pupils       22332
prices.others        5667
dtype: int64

## Coverage: does every canteen have enough data?
I requested one Wednesday per week from 2024-10-01 to 2026-10-01.

In [5]:
all_dates = pd.date_range("2024-10-01", "2026-10-01", freq="W-WED")
total_dates = len(all_dates)
print("Dates requested per canteen:", total_dates)

Dates requested per canteen: 105


In [6]:
coverage = (
    df.groupby(["city", "canteen_name"])["date"]
      .nunique()
      .reset_index(name="dates_with_meals")
)

coverage["coverage_pct"] = (coverage["dates_with_meals"] / total_dates * 100).round(1)
coverage.sort_values("coverage_pct")

,city,canteen_name,dates_with_meals,coverage_pct
31,Würzburg,"Würzburg, Mensa am Studentenhaus",3,2.9
12,Jena,Mensa Ernst-Abbe-Platz,20,19.0
13,Jena,Mensa Philosophenweg,20,19.0
26,Saarbrücken / Saarland,Cafeteria Hochschule für Musik Saar,37,35.2
6,Düsseldorf,Mensa Kamp-Lintfort,46,43.8
7,Düsseldorf,"Mensa Universitätsstraße, Düsseldorf",46,43.8
1,Berlin,Mensa TU Hardenbergstraße,47,44.8
0,Berlin,"Berlin, Wilhelmstadtschulen",64,61.0
11,Heidelberg,"Heidelberg, Triplex-Mensa am Uniplatz",65,61.9
22,Passau,Cafeteria Nikolakloster Passau,67,63.8


In [7]:
CITIES = ["Heidelberg", "Berlin", "Hamburg", "München", "Leipzig",
          "Würzburg", "Nürnberg", "Dresden", "Köln",
          "Mannheim", "Chemnitz", "Straubing", "Passau", "Regensburg",
          "Düsseldorf", "Stuttgart", "Saarbrücken / Saarland",
          "Dortmund / Nordrhein-Westfalen", "Jena"]

missing_cities = set(CITIES) - set(df["city"].unique())
missing_cities

{'Dortmund / Nordrhein-Westfalen', 'Hamburg', 'München'}

## Decision: which canteens do I keep?
Looking at the coverage table, there is a clear gap: 7 canteens are below 45%, and all others are above 60%.
Jena and Düsseldorf have identical counts for both canteens, which suggests their data stopped at some point.
Hamburg, München and Dortmund returned no data at all.
Two canteens are schools, not universities (Berlin Wilhelmstadtschulen, Straubing ABG Gymnasium).

My rule:
1. Keep canteens with at least 50% coverage, because it falls in the natural gap and removes canteens whose data stopped.
2. Remove school canteens, because this project is about university students.

In [8]:
MIN_COVERAGE = 50
SCHOOL_CANTEENS = ["Berlin, Wilhelmstadtschulen",
                   "Straubing, ABG Anton-Bruckner-Gymnasium"]

good_canteens = coverage[coverage["coverage_pct"] >= MIN_COVERAGE]["canteen_name"]
df_clean = df[df["canteen_name"].isin(good_canteens)]
df_clean = df_clean[~df_clean["canteen_name"].isin(SCHOOL_CANTEENS)]

print("Canteens kept:", df_clean["canteen_name"].nunique())
print("Cities kept:", df_clean["city"].nunique())
print("Meals kept:", len(df_clean))

Canteens kept: 23
Cities kept: 13
Meals kept: 17918
